# 01 — Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import statsmodels.api as sm

In [ ]:
PROJECT_ROOT = Path("../").resolve()

index_path = PROJECT_ROOT / "data" / "processed" / "index_St.parquet"
index_df = pd.read_parquet(index_path)

print("index_St head:")
display(index_df.head())

pivot = index_df.pivot_table(index="year", columns="model", values="S_t_mean", aggfunc="mean")
print("\nS_t_mean pivot (year × model):")
display(pivot)

In [ ]:
figures_dir = PROJECT_ROOT / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

ts = index_df.groupby("year").agg(
    S_t_mean=("S_t_mean", "mean"),
    S_t_std=("S_t_mean", "std"),
    log_RV=("log_RV", "mean")
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(ts["year"], ts["S_t_mean"], color="steelblue", linewidth=1.8, label="S_t_mean")
ax1.fill_between(
    ts["year"],
    ts["S_t_mean"] - ts["S_t_std"],
    ts["S_t_mean"] + ts["S_t_std"],
    color="steelblue", alpha=0.2, label="±1 SD"
)
ax1.set_xlabel("Year")
ax1.set_ylabel("S_t_mean", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(ts["year"], ts["log_RV"], color="firebrick", linewidth=1.8, linestyle="--", label="log_RV")
ax2.set_ylabel("log_RV", color="firebrick")
ax2.tick_params(axis="y", labelcolor="firebrick")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Triviality Index S_t and log_RV over Time", fontsize=14)
ax1.grid(True, alpha=0.3)
plt.tight_layout()

out_path = figures_dir / "results_ts.png"
plt.savefig(out_path, dpi=150)
print(f"Saved to {out_path}")
plt.show()

In [ ]:
reg_path = PROJECT_ROOT / "outputs" / "tables" / "regression_results.csv"
reg_df = pd.read_csv(reg_path)

styled = reg_df.style.format(precision=4).set_caption("Panel / TS Regression Results")
display(styled)

In [ ]:
placebo_path = PROJECT_ROOT / "outputs" / "tables" / "placebo_results.csv"
placebo_df = pd.read_csv(placebo_path)

display(placebo_df)

In [ ]:
denom_groups = index_df["denomination"].unique()
cmap = plt.get_cmap("tab10")
color_map = {d: cmap(i) for i, d in enumerate(denom_groups)}

fig, ax = plt.subplots(figsize=(9, 6))
for denom, grp in index_df.groupby("denomination"):
    ax.scatter(
        grp["S_t_mean"], grp["log_RV"],
        label=denom, alpha=0.5, s=18,
        color=color_map[denom]
    )

ax.set_xlabel("S_t_mean")
ax.set_ylabel("log_RV")
ax.set_title("S_t_mean vs log_RV by Denomination", fontsize=14)
ax.legend(title="Denomination", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
scores_path = PROJECT_ROOT / "data" / "processed" / "scores.parquet"
scores = pd.read_parquet(scores_path)

scores["decade"] = (scores["year"] // 10) * 10

drift_pivot = scores.pivot_table(
    index="idiom", columns="decade", values="score_drift", aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(14, max(4, len(drift_pivot) * 0.4)))
im = ax.imshow(
    drift_pivot.values,
    aspect="auto", cmap="RdBu_r",
    vmin=-np.nanmax(np.abs(drift_pivot.values)),
    vmax=np.nanmax(np.abs(drift_pivot.values))
)

ax.set_xticks(range(len(drift_pivot.columns)))
ax.set_xticklabels(drift_pivot.columns.astype(str), rotation=45, ha="right")
ax.set_yticks(range(len(drift_pivot.index)))
ax.set_yticklabels(drift_pivot.index)
ax.set_title("Decade-Level Centroid Drift (idiom × decade)", fontsize=14)
ax.set_xlabel("Decade")
ax.set_ylabel("Idiom")

plt.colorbar(im, ax=ax, label="score_drift")
plt.tight_layout()
plt.show()

## Interpretation

Add narrative here.